# Redfield Relaxation Tests

This notebook contains comprehensive tests for Redfield relaxation operations. It validates:
1. **Redfield vs Lindblad consistency** - Verifies that Redfield relaxation under constant spectral density matches Lindblad formalism
2. **Context operations** - Tests multiplication, concatenation, and summation with Redfield relaxation channels
3. **Superoperator correctness** - Validates that relaxation superoperators are properly constructed and transformed

In [111]:
%load_ext autoreload
%autoreload 2

import sys
import os
import math
from importlib import reload

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))



import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [112]:
dtype = torch.float64
device = torch.device("cpu")

complex_dtype = torch.complex128 if dtype == torch.float64 else torch.complex64

In [113]:
import torch
import numpy as np
import math
from typing import Optional, List, Tuple
from mars.population.contexts import Context, SummedContext
from mars.population import transform, RedfieldRelaxationChannel
from mars import spin_model
import mars

# 1. Create Samples for Checking

In [114]:
def create_samples():
    g_tensor_1 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_1 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)  # D=200 MHz, E=50 MHz
    g_tensor_2 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_2 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)
    
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_1],
        electron_electron=[(0, 0, zfs_1)],
        dtype=dtype,
    )
    sample_1 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )


    g_tensor = spin_model.Interaction((2.02, 2.14, 2.16), dtype=dtype)
    zfs = spin_model.DEInteraction((200 * 1e6, 70 * 1e6), dtype=dtype)
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_2],
        electron_electron=[(0, 0, zfs_2)],
        dtype=dtype
    )
    sample_2 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
)
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0, 1.0],
        g_tensors=[g_tensor_1, g_tensor_2],
        electron_electron=[(0, 0, zfs_1), (1, 1, zfs_2), (1, 0, zfs_1)],
        dtype=dtype
    )
    sample_3 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
)
    return sample_1, sample_2, sample_3

def get_eigen_basis(sample, field: float):
    magnetic_field = torch.tensor(field)
    F, _, _, Gz = sample.get_hamiltonian_terms()
    H = F + Gz * magnetic_field
    H = H.unsqueeze(-3)
    values, vectors = torch.linalg.eigh(H)
    return vectors

In [115]:
def create_operator(sample: int, operator_dtype = complex_dtype):
    N = sample.spin_system_dim
    operator = torch.randn((N, N), dtype=operator_dtype)
    operator = (operator + operator.transpose(-2, -1).conj()) / 2
    return operator

# 2. Redfield vs Lindblad Superoperator Consistency Test

This test verifies that the Redfield relaxation superoperator matches the Lindblad formalism when using a constant spectral density function and secular approximation. Also for non-secular approximation it checkes correctness of secular terms. We create two contexts:
- One with **RedfieldRelaxationChannel**
- One with **Lindblad superoperator** constructed via `transform.lindblad_dissipator_from_operator()`

Both contexts are compared using `get_transformed_driven_superop()` and `get_transformed_driven_rates()`.

In [116]:
def constant_spectral_density(omega_rad_s: torch.Tensor) -> torch.Tensor:
    """Constant spectral density function for Lindblad comparison."""
    return torch.ones_like(omega_rad_s)

def random_spectral_density(
        omega_rad_s: torch.Tensor, 
        diagonal_value: float = 1.2,
        off_diagonal_value: float = None,
        random_off_diagonal: bool = False,
        seed: int = None
    ) -> torch.Tensor:
    """Constant spectral density function for Lindblad comparison."""
    if random_off_diagonal and seed is not None:
        torch.manual_seed(seed)
    
    if random_off_diagonal:
        density = torch.randn_like(omega_rad_s)
    else:
        density = torch.ones_like(omega_rad_s)
        if off_diagonal_value is not None:
            density = density * off_diagonal_value
    
    if density.dim() == 2 and density.shape[0] == density.shape[1]:
        torch.diagonal(density).fill_(diagonal_value)
    
    return density


def check_redfield_vs_lindblad_superoperator_rates(
    context_redfield: Context,
    context_lindblad: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    eigen_basis: bool,
    atol: float = 1e-4
) -> None:
    """
    Compare Redfield Superoperator with Lindblad Superoperator.
    
    Both contexts are queried using get_transformed_driven_superop() and 
    get_transformed_driven_probs() to verify consistency.
    
    Tests performed:
    1. Population block comparison (diagonal density matrix elements)
    2. Pure dephasing rates (coherence block diagonals)
    3. Transition rates consistency
    4. Coherence decay rates (should be negative)
    """
    dim = context_redfield.spin_system_dim
    dim_sq = dim * dim
    
    # 1. Get Redfield driven superoperator
    R_redfield = context_redfield.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # 2. Get Lindblad driven superoperator
    R_lindblad = context_lindblad.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    if eigen_basis:
        N = R_redfield.shape[-1]
        batch_shape = R_redfield.shape[:-2]  # Get all batch dimensions
        R_lindblad = R_lindblad.unsqueeze(0).expand(batch_shape + (N, N))
    
    assert R_redfield is not None and R_lindblad is not None, \
        "Superoperators should not be None"
    assert R_redfield.shape == R_lindblad.shape, \
        f"Superoperator shape mismatch: {R_redfield.shape} vs {R_lindblad.shape}"
    
    # =========================================================================
    # TEST 1: Population block comparison (diagonal density matrix elements)
    # =========================================================================
    pop_indices = torch.arange(dim, device=R_redfield.device)
    pop_idx_flat = pop_indices * dim + pop_indices
    R_pop_red = R_redfield[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    R_pop_lin = R_lindblad[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    
    assert torch.allclose(R_pop_red, R_pop_lin, atol=atol), \
        "Population block of Redfield superoperator does not match Lindblad"
    
    # =========================================================================
    # TEST 2: Pure dephasing elements (coherence block diagonals)
    # =========================================================================

    coh_indices = []
    for a in range(dim):
        for b in range(dim):
            if a != b:
                coh_indices.append(a * dim + b)
    
    if coh_indices:
        coh_idx_tensor = torch.tensor(coh_indices, dtype=torch.long, device=R_redfield.device)
        
        R_coh_red_diag = R_redfield[..., coh_idx_tensor, coh_idx_tensor]
        R_coh_lin_diag = R_lindblad[..., coh_idx_tensor, coh_idx_tensor]

        
        assert torch.allclose(R_coh_red_diag, R_coh_lin_diag, atol=atol), \
            f"Pure dephasing rates mismatch. Max diff: {torch.abs(R_coh_red_diag - R_coh_lin_diag).max().item():.2e}"
        
        assert torch.all(R_coh_red_diag.real <= 1e-10), \
            f"Redfield dephasing rates should be negative. Max: {R_coh_red_diag.real.max().item():.2e}"
        assert torch.all(R_coh_lin_diag.real <= 1e-10), \
            f"Lindblad dephasing rates should be negative. Max: {R_coh_lin_diag.real.max().item():.2e}"
        
        # Check that dephasing rates are related to population rates
        # For secular Redfield: Gamma_ab = 0.5 * (W_a + W_b) + W_pure_dephasing
        pop_decay_rates_red = R_pop_red.sum(dim=-1)
        pop_decay_rates_lin = R_pop_lin.sum(dim=-1)
        
        expected_min_dephasing = 0.5 * (pop_decay_rates_red[..., [coh // dim for coh in coh_indices]] + 
                                        pop_decay_rates_red[..., [coh % dim for coh in coh_indices]])
        
        assert torch.all(R_coh_red_diag.real <= -expected_min_dephasing.real + atol), \
            "Dephasing rates should be at least half the sum of population decay rates"
    
    # =========================================================================
    # TEST 3: Get driven rates from both contexts
    # =========================================================================
    rates_redfield = context_redfield.get_transformed_driven_probs(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    rates_lindblad = context_lindblad.get_transformed_driven_probs(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # Compare rates
    if rates_redfield is not None and rates_lindblad is not None:
        assert torch.allclose(rates_redfield, rates_lindblad, atol=atol), \
            "Redfield rates do not match Lindblad rates"
    
    # =========================================================================
    # TEST 4: Check coherence decay (should be negative)
    # =========================================================================
    if coh_indices:
        coh_idx_tensor = torch.tensor(coh_indices, dtype=torch.long, device=R_redfield.device)
        R_coh_red_diag = R_redfield[..., coh_idx_tensor, coh_idx_tensor]
        assert torch.all(R_coh_red_diag.real <= 1e-10), \
            f"Coherence decay rates should be negative. Max: {R_coh_red_diag.real.max().item():.2e}"
    
    # =========================================================================
    # TEST 5: Verify trace preservation for both superoperators
    # =========================================================================
    trace_red = R_redfield[..., :, range(0, dim_sq, dim+1)].sum(dim=-1)
    trace_lin = R_lindblad[..., :, range(0, dim_sq, dim+1)].sum(dim=-1)
    
    assert torch.allclose(trace_red, torch.zeros_like(trace_red), atol=atol), \
        "Redfield superoperator doesn't preserve trace"
    assert torch.allclose(trace_lin, torch.zeros_like(trace_lin), atol=atol), \
        "Lindblad superoperator doesn't preserve trace"
    
    print("✓ Redfield vs Lindblad superoperator consistency test passed")
    print(f"  - Population block: ✓")
    print(f"  - Pure dephasing elements: ✓")
    print(f"  - Transition rates: ✓")
    print(f"  - Coherence decay: ✓")
    print(f"  - Trace preservation: ✓")


def check_redfield_vs_lindblad_superoperator_direct(
    context_redfield: Context,
    context_lindblad: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    eigen_basis: bool,
    atol: float = 1e-4
) -> None:
    """
    Compare Redfield Superoperator with Lindblad Superoperator.
    
    Both contexts are queried using get_transformed_driven_superop() and 
    get_transformed_driven_probs() to verify consistency.
    """
    dim = context_redfield.spin_system_dim
    dim_sq = dim * dim

    # 1. Get Redfield driven superoperator
    R_redfield = context_redfield.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )

    # 2. Get Lindblad driven superoperator
    R_lindblad = context_lindblad.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )

    if eigen_basis:
        N = R_redfield.shape[-1]
        batch_shape = R_redfield.shape[:-2]  # Get all batch dimensions
        R_lindblad = R_lindblad.unsqueeze(0).expand(batch_shape + (N, N))

    # 3. Compare superoperators
    assert R_redfield is not None and R_lindblad is not None, \
        "Superoperators should not be None"

    assert R_redfield.shape == R_lindblad.shape, \
        f"Superoperator shape mismatch: {R_redfield.shape} vs {R_lindblad.shape}"

    assert torch.allclose(R_redfield, R_lindblad, atol=atol), \
        "R_red and R_lin are not the same"
    
    print("✓ Redfield vs Lindblad superoperator secular equality test passed")

In [117]:
def create_redfield_context(sample, operators: list[torch.Tensor], basis="zfs", enforce_secularity=True, dtype=torch.float64):
    """Create a Context with RedfieldRelaxationChannel."""
    operator_components = [(op, None) for op in operators]
    
    redfield_channel = RedfieldRelaxationChannel(
        operator_components=operator_components,
        spectral_density_func=random_spectral_density,
        thermal_balance_mode="skip"
    )
    
    context = Context(
        basis=basis,
        sample=sample,
        init_populations=[0.5, 0.35, 0.15][:sample.spin_system_dim],
        redfield_channels=[redfield_channel],
        enforce_secularity=enforce_secularity,
        dtype=dtype
    )
    
    return context


def create_lindblad_context(sample, operators: list[torch.Tensor], enforce_secularity=True, basis="zfs", dtype=torch.float64):
    """Create a Context with Lindblad superoperator from operator."""
    dim = sample.spin_system_dim
    dim_sq = dim * dim

    L_superop = sum([
        transform.Liouvilleator().lindblad_dissipator_from_operator(L_op)
        for L_op in operators
    ])
    L_superop = L_superop
        
    context = Context(
        basis=basis,
        sample=sample,
        init_populations=[0.5, 0.35, 0.15][:dim],
        relaxation_superop=L_superop,  # Use population block for comparison
        enforce_secularity=enforce_secularity,
        dtype=dtype
    )
    return context

# 3. Context Operations with Redfield Relaxation

This section tests that Redfield relaxation contexts work correctly with:
- **Multiplication (@)** - Tensor product of independent subsystems
- **Concatenation (mars.concat)** - Block-diagonal composition for different species
- **Summation (+)** - Parallel relaxation mechanisms

In [124]:
def check_redfield_multiplication_correctness(
    context_1: Context,
    context_2: Context,
    full_system_vectors_1: torch.Tensor,
    full_system_vectors_2: torch.Tensor,
    fields: torch.Tensor,
    energies_1: torch.Tensor,
    energies_2: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-6,
    rtol: float = 1e-5
) -> None:
    """
    Validate Kronecker product (@) of contexts with Redfield relaxation.
    
    Physical rules tested:
    - Combined superoperator: R_total = R₁ ⊗ I + I ⊗ R₂
    - Combined energies: E_total[a,b] = E₁[a] + E₂[b] (ADD, not Kronecker)
    - Relaxation channels combine correctly for independent subsystems
    """
    full_system_vectors_tot = transform.batched_kron(
        full_system_vectors_1, full_system_vectors_2
    )
    mul_context = context_1 @ context_2

    # 1. Dimensionality check
    dim1 = context_1.spin_system_dim
    dim2 = context_2.spin_system_dim
    dim_tot = mul_context.spin_system_dim
    assert dim_tot == dim1 * dim2, \
        f"Dimension mismatch: {dim1} ⊗ {dim2} = {dim_tot} (expected {dim1 * dim2})"

    energies_tot = energies_1.unsqueeze(-1) + energies_2.unsqueeze(-2)  # [..., N1, N2]
    energies_tot = energies_tot.reshape(*energies_1.shape[:-1], -1)  # [..., N1*N2]
    
    # 3. Check relaxation superoperators combine correctly
    superop1 = context_1.get_transformed_driven_superop(
        full_system_vectors_1, fields=fields, energies=energies_1, temperature=temperature
    )
    superop2 = context_2.get_transformed_driven_superop(
        full_system_vectors_2, fields=fields, energies=energies_2, temperature=temperature
    )
    superop_tot = mul_context.get_transformed_driven_superop(
        full_system_vectors_tot, fields=fields, energies=energies_tot, temperature=temperature
    )
    
    dim1_sq = dim1 * dim1
    dim2_sq = dim2 * dim2
    I1_liouv = torch.eye(dim1_sq, device=superop1.device, dtype=superop1.dtype)
    I2_liouv = torch.eye(dim2_sq, device=superop2.device, dtype=superop2.dtype)

    batch_shape = torch.broadcast_shapes(superop1.shape[:-2], superop2.shape[:-2])
    superop1_exp = superop1.expand(batch_shape + (dim1_sq, dim1_sq))
    superop2_exp = superop2.expand(batch_shape + (dim2_sq, dim2_sq))
    I1_exp = I1_liouv.expand(batch_shape + (dim1_sq, dim1_sq))
    I2_exp = I2_liouv.expand(batch_shape + (dim2_sq, dim2_sq))

    expected_superop = transform.batched_kron(superop1_exp, I2_exp) + \
                       transform.batched_kron(I1_exp, superop2_exp)

    expected_superop = transform.reshape_superoperator_tensor_to_kronecker_basis(
        expected_superop, subsystem_dims=[dim1, dim2]
    )
    
    assert torch.allclose(superop_tot, expected_superop, atol=atol, rtol=rtol), \
        f"Redfield superoperators don't combine via Liouville Kronecker sum. Max diff: {torch.abs(superop_tot - expected_superop).max().item():.2e}"
    
    # 4. Check rates combine correctly

    rates_tot = mul_context.get_transformed_driven_probs(
        full_system_vectors_tot, fields=fields, energies=energies_tot, temperature=temperature
    )
    
    pop_indices = torch.arange(9, device=expected_superop.device)
    pop_idx_flat = pop_indices * 9 + pop_indices
    pop_from_super = expected_superop[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    
    indices = torch.arange(9, device=pop_indices.device)
    rates_tot[..., indices, indices] = -rates_tot.sum(dim=-2)
    
    assert torch.allclose(rates_tot, pop_from_super.real, atol=atol, rtol=rtol), \
        f"Redfield rates don't combine correctly. Max diff: {torch.abs(rates_tot - expected_rates).max().item():.2e}"
    
    print("✓ Redfield multiplication test passed: dimensions, energies, superoperators, and rates")


def check_redfield_concatenation_correctness(
    context_1: Context,
    context_2: Context,
    full_system_vectors_1: torch.Tensor,
    full_system_vectors_2: torch.Tensor,
    fields: torch.Tensor,
    energies_1: torch.Tensor,
    energies_2: torch.Tensor,
    energies_cat: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-6,
    rtol: float = 1e-5
) -> None:
    """
    Validate concatenation of contexts with Redfield relaxation.
    
    Physical interpretation: Different molecular species with independent relaxation.
    Mathematical operation: Direct sum (block-diagonal composition).
    """
    def direct_sum_batched(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        n = A.shape[-1]
        m = B.shape[-1]
        batch_size = A.shape[:-2]
        total_dim = n + m
        
        result = torch.zeros(*batch_size, total_dim, total_dim, 
                             dtype=A.dtype, device=A.device)
        result[..., :n, :n] = A
        result[..., n:, n:] = B
        return result
    
    concat_context = mars.concat([context_1, context_2])
    full_system_vectors = direct_sum_batched(full_system_vectors_1, full_system_vectors_2)
    
    # 1. Dimensionality (direct sum: N_total = N₁ + N₂)
    dim1 = context_1.spin_system_dim
    dim2 = context_2.spin_system_dim
    dim_concat = concat_context.spin_system_dim
    assert dim_concat == dim1 + dim2, \
        f"Dimension mismatch: {dim1} ⊕ {dim2} = {dim_concat} (expected {dim1 + dim2})"
    
    superop1 = context_1.get_transformed_driven_superop(
        full_system_vectors_1, 
        fields=fields, 
        energies=energies_1, 
        temperature=temperature
    )
    superop2 = context_2.get_transformed_driven_superop(
        full_system_vectors_2, 
        fields=fields, 
        energies=energies_2, 
        temperature=temperature
    )
    superop_concat = concat_context.get_transformed_driven_superop(
        full_system_vectors, 
        fields=fields, 
        energies=torch.cat([energies_1, energies_2], dim=-1),
        temperature=temperature
    )
    
    if superop1 is not None and superop2 is not None and superop_concat is not None:
        dim1_sq = dim1 * dim1
        dim2_sq = dim2 * dim2
    
        expected_superop = transform.reshape_superoperators_list_to_direct_sum_basis([superop1, superop2])
        dim_1 = context_1.spin_system_dim
        dim_2 = context_2.spin_system_dim
        expected_superop_1, expected_superop_2 = transform.reshape_direct_sum_basis_to_superoperators_list(superop_concat, (dim_1, dim_2))

        assert torch.allclose(expected_superop_1, superop1, atol=atol, rtol=rtol), \
            f"Redfield superoperators don't form block-diagonal structure. Max diff: {torch.abs(expected_superop_1 - superop1).max().item():.2e}"
                
        assert torch.allclose(expected_superop_2, superop2, atol=atol, rtol=rtol), \
            f"Redfield superoperators don't form block-diagonal structure. Max diff: {torch.abs(expected_superop_1 - superop2).max().item():.2e}"
    
    # 3. Check rates form block-diagonal structure
    rates1 = context_1.get_transformed_driven_probs(
        full_system_vectors_1, 
        fields=fields, 
        energies=energies_1, 
        temperature=temperature
    )
    rates2 = context_2.get_transformed_driven_probs(
        full_system_vectors_2, 
        fields=fields, 
        energies=energies_2, 
        temperature=temperature
    )
    rates_concat = concat_context.get_transformed_driven_probs(
        full_system_vectors, 
        fields=fields, 
        energies=energies_cat,
        temperature=temperature
    )
    
    if rates1 is not None and rates2 is not None and rates_concat is not None:
        expected_rates = torch.zeros(
            rates1.shape[:-2]+ (dim1 + dim2,) + (dim1 + dim2,),
            device=rates1.device, dtype=rates1.dtype
        )
        expected_rates[..., :dim1, :dim1] = rates1
        expected_rates[..., dim1:, dim1:] = rates2
        print(rates_concat.shape)
        print(expected_rates.shape)
        
        assert torch.allclose(rates_concat, expected_rates, atol=atol, rtol=rtol), \
            f"Redfield rates don't form block-diagonal structure. Max diff: {torch.abs(rates_concat - expected_rates).max().item():.2e}"
    
    print("✓ Redfield concatenation test passed: block-diagonal structure")


def check_redfield_sum_correctness(
    context_1: Context,
    context_2: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-6,
    rtol: float = 1e-5
) -> None:
    """
    Validate addition (+) of contexts with Redfield relaxation.
    
    Physical rules tested:
    - Parallel relaxation mechanisms sum element-wise
    - Superoperators sum correctly
    """
    sum_context = context_1 + context_2
    
    # 1. Dimensionality preserved
    dim1 = context_1.spin_system_dim
    dim2 = context_2.spin_system_dim
    dim_sum = sum_context.spin_system_dim
    assert dim1 == dim2 == dim_sum, \
        f"Dimension mismatch in sum: {dim1} + {dim2} = {dim_sum}"
    
    # 2. Superoperators sum
    superop1 = context_1.get_transformed_driven_superop(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    superop2 = context_2.get_transformed_driven_superop(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    superop_sum = sum_context.get_transformed_driven_superop(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    
    if superop1 is not None and superop2 is not None and superop_sum is not None:
        expected_superop = superop1 + superop2
        assert torch.allclose(superop_sum, expected_superop, atol=atol, rtol=rtol), \
            "Redfield superoperators don't sum correctly"
    
    # 3. Rates sum
    rates1 = context_1.get_transformed_driven_probs(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    rates2 = context_2.get_transformed_driven_probs(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    rates_sum = sum_context.get_transformed_driven_probs(
        full_system_vectors, fields=fields, energies=energies, temperature=temperature
    )
    
    if rates1 is not None and rates2 is not None and rates_sum is not None:
        expected_rates = rates1 + rates2
        assert torch.allclose(rates_sum, expected_rates, atol=atol, rtol=rtol), \
            "Redfield rates don't sum correctly"
    
    print("✓ Redfield sum test passed: superoperators and rates sum correctly")

# 4. Secular Approximation

Check that Redfield under secular approximation equalt to non-secular Redfield with secular mask

In [125]:
def get_secular_mask(spin_system_dim: int):
    """
    Secular approximation mask:
    - Populations can transfer between each other
    - Each coherence evolves independently
    
    Args:
        spin_system_dim: Hilbert space dimension N
    
    Returns:
        Boolean mask of shape (N², N²)
    """
    N = spin_system_dim
    dim_liouville = N * N
    mask = torch.eye(dim_liouville, dtype=torch.bool)
    pop_indices = torch.arange(0, dim_liouville, N + 1)
    mask[pop_indices[:, None], pop_indices[None, :]] = True
    return mask

def check_secularity_from_context(
    context_secular: Context,
    context_non_secular: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-6,
    rtol: float = 1e-5
) -> None:
    """
    Compare secular vs non-secular Redfield relaxation contexts.
    
    Physical rules tested:
    - Secular approximation removes off-diagonal coherence coupling terms
    - Population transfer rates should be similar (secular is subset of non-secular)
    - Non-secular may have additional coherence-coherence coupling terms
    - Both should preserve trace and Hermiticity
    """
    dim = context_secular.spin_system_dim
    dim_sq = dim * dim
    
    # 1. Get superoperators from both contexts
    R_secular = context_secular.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    R_non_secular = context_non_secular.get_transformed_driven_superop(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    assert R_secular is not None and R_non_secular is not None, \
        "Superoperators should not be None"
    assert R_secular.shape == R_non_secular.shape, \
        f"Superoperator shape mismatch: {R_secular.shape} vs {R_non_secular.shape}"
    
    mask = get_secular_mask(context_secular.spin_system_dim)
    R_non_secular_mask = R_non_secular * mask.type_as(R_non_secular)
    
    assert torch.allclose(R_non_secular_mask.imag, torch.zeros_like(R_non_secular_mask.imag), atol=atol, rtol=rtol), \
        "Secularity Imaganary should be zero" 
    
    assert torch.allclose(R_non_secular_mask, R_secular, atol=atol, rtol=rtol), \
        "Secularity failed" 
    
    # 2. Get rates from both contexts
    rates_secular = context_secular.get_transformed_driven_probs(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    rates_non_secular = context_non_secular.get_transformed_driven_probs(
        full_system_vectors=full_system_vectors,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # 3. Population block comparison (should be similar)
    pop_indices = torch.arange(dim, device=R_secular.device)
    pop_idx_flat = pop_indices * dim + pop_indices
    
    R_pop_sec = R_secular[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    R_pop_non = R_non_secular[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    
    # Population rates should be close (secular approximation primarily affects coherences)
    assert torch.allclose(R_pop_sec.real, R_pop_non.real, atol=atol, rtol=rtol), \
        "Population blocks differ significantly between secular and non-secular"

def check_secularity(dtype=torch.float64) -> None:
    """Run secularity comparison tests for Redfield relaxation."""
    print("=" * 60)
    print("Starting Redfield Secularity Tests")
    print("=" * 60)
    
    sample_1, sample_2, sample_3 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 0.1)
    vectors_2 = get_eigen_basis(sample_2, 0.1)
    
    # Test parameters
    fields = torch.tensor([0.1], dtype=dtype)
    temperature = torch.tensor(300.0, dtype=dtype)
    
    # Get energies from Hamiltonian
    F, _, _, Gz = sample_1.get_hamiltonian_terms()
    H = F + Gz * fields
    H = H.unsqueeze(-3)
    energies, _ = torch.linalg.eigh(H)
    
    # =========================================================================
    # TEST: Secular vs Non-Secular Redfield Relaxation
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST: Secular vs Non-Secular Redfield Relaxation")
    print("=" * 60)
    
    basis = "eigen"
    
    # Create operators for Lindblad comparison (optional)
    operators = [create_operator(sample_1, complex_dtype) for _ in range(3)]
    
    # Create secular context
    context_secular = create_redfield_context(
        sample_1, 
        operators=operators, 
        enforce_secularity=True, 
        basis=basis, 
        dtype=dtype
    )
    
    # Create non-secular context
    context_non_secular = create_redfield_context(
        sample_1, 
        operators=operators, 
        enforce_secularity=False, 
        basis=basis, 
        dtype=dtype
    )
    
    # Run secularity comparison
    check_secularity_from_context(
        context_secular,
        context_non_secular,
        full_system_vectors=vectors_1,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # =========================================================================
    # TEST: Secular Approximation at Different Field Strengths
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST: Secular Approximation at Different Field Strengths")
    print("=" * 60)
    
    field_strengths = [0.01, 0.1, 0.5, 1.0]  # Tesla
    operators = [create_operator(sample_1, complex_dtype) for _ in range(3)]   # 3 is a just magic number. 
    
    for field in field_strengths:
        fields_test = torch.tensor([field], dtype=dtype)
        
        # Get energies at this field
        H_test = F + Gz * fields_test
        H_test = H_test.unsqueeze(-3)
        energies_test, _ = torch.linalg.eigh(H_test)
        
        # Get eigenbasis at this field
        vectors_test = get_eigen_basis(sample_1, field)
        
        # Create contexts
        ctx_sec = create_redfield_context(
            sample_1, 
            operators=operators,
            enforce_secularity=True, 
            basis=basis, 
            dtype=dtype
        )
        ctx_non = create_redfield_context(
            sample_1, 
            operators=operators,
            enforce_secularity=False, 
            basis=basis, 
            dtype=dtype
        )
        
        # Get superoperators
        R_sec = ctx_sec.get_transformed_driven_superop(
            full_system_vectors=vectors_test,
            fields=fields_test,
            energies=energies_test,
            temperature=temperature
        )
        
        R_non = ctx_non.get_transformed_driven_superop(
            full_system_vectors=vectors_test,
            fields=fields_test,
            energies=energies_test,
            temperature=temperature
        )
        
        # Compare population blocks
        dim = sample_1.spin_system_dim
        pop_indices = torch.arange(dim, device=R_sec.device)
        pop_idx_flat = pop_indices * dim + pop_indices
        
        R_pop_sec = R_sec[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
        R_pop_non = R_non[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
        
        pop_diff = torch.abs(R_pop_sec - R_pop_non).max().item()
                
        # At high fields, secular approximation should be better
        if field >= 0.5:
            assert pop_diff < 1e-4, \
                f"At high field ({field} T), secular and non-secular should match closely. Diff: {pop_diff:.2e}"
    
    print("✓ Field strength secularity test passed")
    
    # =========================================================================
    # Summary
    # =========================================================================
    print("\n" + "=" * 60)
    print("✅ All Redfield Secularity Tests Passed Successfully!")
    print("=" * 60)
    print("\nTests completed:")
    print("  1. ✓ Secular vs non-secular superoperator structure")
    print("  2. ✓ Population rates consistency")
    print("  3. ✓ Coherence coupling differences")
    print("  4. ✓ Field strength dependence")
    print("=" * 60)

# 4. Run Comprehensive Tests

This cell executes all Redfield relaxation tests including:
1. Redfield vs Lindblad consistency
2. Context multiplication with relaxation
3. Context concatenation with relaxation
4. Context summation with relaxation

In [127]:
def test_comprehensive_redfield_operations(dtype=torch.float64):
    """Run all Redfield relaxation tests."""
    print("=" * 60)
    print("Starting Redfield Relaxation Comprehensive Tests")
    print("=" * 60)
    
    # =========================================================================
    # TEST 0: Redfield Secularity
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 0:Redfield Secularity")
    print("=" * 60)
    check_secularity()
    
    sample_1, sample_2, sample_3 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 0.1)
    vectors_2 = get_eigen_basis(sample_2, 0.1)
    
    # Test parameters
    fields = torch.tensor([0.1], dtype=dtype)
    temperature = torch.tensor(300.0, dtype=dtype)
    
    # Get energies from Hamiltonian for BOTH samples
    F1, _, _, Gz1 = sample_1.get_hamiltonian_terms()
    H1 = F1 + Gz1 * fields
    H1 = H1.unsqueeze(-3)
    energies_1, _ = torch.linalg.eigh(H1)
    
    F2, _, _, Gz2 = sample_2.get_hamiltonian_terms()
    H2 = F2 + Gz2 * fields
    H2 = H2.unsqueeze(-3)
    energies_2, _ = torch.linalg.eigh(H2)
    
    sample_cat = mars.concat((sample_1, sample_2))
    F_cat, _, _, Gz_cat = sample_cat.get_hamiltonian_terms()
    H_cat = F_cat + Gz_cat * fields
    H_cat = H_cat.unsqueeze(-3)
    energies_cat, _ = torch.linalg.eigh(H_cat)

    
    
    # =========================================================================
    # TEST 1: Redfield vs Lindblad Superoperator Consistency
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 1: Redfield vs Lindblad Superoperator Consistency")
    print("=" * 60)
    
    basis = "eigen"
    eigen_bais = True
    
    operators = [create_operator(sample_1, complex_dtype) for _ in range(3)]   # 3 is a just magic number. 
    context_redfield = create_redfield_context(sample_1, operators, enforce_secularity=False, basis=basis, dtype=dtype)
    context_lindblad = create_lindblad_context(sample_1, operators, enforce_secularity=False, basis=basis, dtype=dtype)
    
    check_redfield_vs_lindblad_superoperator_rates(
        context_redfield, context_lindblad, vectors_1, 
        fields.unsqueeze(-2), energies_1, temperature, eigen_bais
    )
    
    basis = "eigen"
    eigen_bais = True
    
    operators = [create_operator(sample_1, complex_dtype) for _ in range(3)]   # 3 is a just magic number. 
    context_redfield = create_redfield_context(sample_1, operators, enforce_secularity=True, basis=basis, dtype=dtype)
    context_lindblad = create_lindblad_context(sample_1, operators, enforce_secularity=True, basis=basis, dtype=dtype)
    
    check_redfield_vs_lindblad_superoperator_direct(
        context_redfield, context_lindblad, vectors_1, 
        fields.unsqueeze(-2), energies_1, temperature, eigen_bais
    )
    
        
    basis = "zfs"
    eigen_bais = False
    operators = [create_operator(sample_1, complex_dtype) for _ in range(3)] 
    # 3 is a just magic number. 
    context_redfield = create_redfield_context(sample_1, operators, enforce_secularity=False, basis=basis, dtype=dtype)
    context_lindblad = create_lindblad_context(sample_1, operators, enforce_secularity=False, basis=basis, dtype=dtype)
    
    check_redfield_vs_lindblad_superoperator_rates(
        context_redfield, context_lindblad, vectors_1, 
        fields.unsqueeze(-2), energies_1, temperature, eigen_bais
    )
    
    # =========================================================================
    # TEST 2: Context Multiplication with Redfield Relaxation
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 2: Context Multiplication with Redfield Relaxation")
    print("=" * 60)
    
    operators_1 = [create_operator(sample_1, complex_dtype) for _ in range(3)]
    context_1 = create_redfield_context(
        sample_1, operators_1, enforce_secularity=False, basis="zfs", dtype=dtype
    )
    operators_2 = [create_operator(sample_2, complex_dtype) for _ in range(3)]
    context_2 = create_redfield_context(
        sample_2, operators_2, enforce_secularity=False, basis="zfs", dtype=dtype
    )
    

    
    check_redfield_multiplication_correctness(
        context_1, context_2, 
        vectors_1, vectors_2, 
        fields.unsqueeze(-2), 
        energies_1, energies_2,
        temperature
    )
    
    # =========================================================================
    # TEST 3: Context Concatenation with Redfield Relaxation
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 3: Context Concatenation with Redfield Relaxation")
    print("=" * 60)
    
    check_redfield_concatenation_correctness(
        context_1, context_2, 
        vectors_1, vectors_2,
        fields.unsqueeze(-2),
        energies_1, energies_2, energies_cat,
        temperature
    )
    
    # =========================================================================
    # TEST 4: Context Summation with Redfield Relaxation
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 4: Context Summation with Redfield Relaxation")
    print("=" * 60)
    
    # Create two contexts on same sample for summation test
    context_sum_1 = create_redfield_context(sample_1, operators, basis="zfs", dtype=dtype)
    context_sum_2 = create_redfield_context(sample_1, operators, basis="zfs", dtype=dtype)
    
    check_redfield_sum_correctness(
        context_sum_1, context_sum_2, vectors_1,
        fields.unsqueeze(-2), energies_1, temperature
    )
    
    # =========================================================================
    # TEST 5: Lindblad Dissipator from Operator
    # =========================================================================
    print("\n" + "=" * 60)
    print("TEST 5: Lindblad Dissipator from Operator")
    print("=" * 60)
    
    dim = sample_1.spin_system_dim
    dim_sq = dim * dim
    
    # Test with different Lindblad operators
    for i in range(min(3, dim)):
        L_op = torch.zeros((dim, dim), dtype=complex_dtype)
        L_op[i, i] = 1.0
        
        L_superop = transform.Liouvilleator().lindblad_dissipator_from_operator(L_op.unsqueeze(0))
        
        # Verify superoperator properties
        assert L_superop.shape[-2:] == (dim_sq, dim_sq), \
            f"Operator {i}: Superoperator has wrong shape"
        
        # Check trace preservation
        trace_check = L_superop[..., :, range(0, dim_sq, dim+1)].sum(dim=-1)
        assert torch.allclose(trace_check, torch.zeros_like(trace_check), atol=1e-5), \
            f"Operator {i}: Superoperator doesn't preserve trace"
    
    print("✓ Lindblad dissipator from operator test passed")
    
    # =========================================================================
    # Summary
    # =========================================================================
    print("\n" + "=" * 60)
    print("✅ All Redfield Relaxation Tests Passed Successfully!")
    print("=" * 60)
    print("\nTests completed:")
    print("  1. ✓ Redfield vs Lindblad superoperator consistency")
    print("  2. ✓ Context multiplication with relaxation")
    print("  3. ✓ Context concatenation with relaxation")
    print("  4. ✓ Context summation with relaxation")
    print("  5. ✓ Lindblad dissipator from operator")
    print("=" * 60)


# Run all tests
test_comprehensive_redfield_operations()

Starting Redfield Relaxation Comprehensive Tests

TEST 0:Redfield Secularity
Starting Redfield Secularity Tests

TEST: Secular vs Non-Secular Redfield Relaxation

TEST: Secular Approximation at Different Field Strengths
✓ Field strength secularity test passed

✅ All Redfield Secularity Tests Passed Successfully!

Tests completed:
  1. ✓ Secular vs non-secular superoperator structure
  2. ✓ Population rates consistency
  3. ✓ Coherence coupling differences
  4. ✓ Field strength dependence

TEST 1: Redfield vs Lindblad Superoperator Consistency
✓ Redfield vs Lindblad superoperator consistency test passed
  - Population block: ✓
  - Pure dephasing elements: ✓
  - Transition rates: ✓
  - Coherence decay: ✓
  - Trace preservation: ✓
✓ Redfield vs Lindblad superoperator secular equality test passed
✓ Redfield vs Lindblad superoperator consistency test passed
  - Population block: ✓
  - Pure dephasing elements: ✓
  - Transition rates: ✓
  - Coherence decay: ✓
  - Trace preservation: ✓

TEST 2